# SVM — Propensión de compra de iPhone
---

**Autor:** Borja Mora Mendez
**Contacto:** [borja.mora.mendez@gmail.com](mailto:borja.mora.mendez@gmail.com) · [LinkedIn](https://www.linkedin.com/in/borja-mora-mendez/)
**Repositorio:** [Data Analytics Portfolio](https://github.com/BORJAMOME/Data-Analytics-Portfolio)
**Categoria:** Machine Learning · Supervisado · Clasificacion · Support Vector Machines

---

### Objetivo

Comparar **SVC con diferentes kernels** (linear, polynomial, RBF) para predecir si un cliente comprará un iPhone a partir de sus ingresos y su fidelidad tecnológica. Se busca el kernel que mejor separa a compradores de no compradores.

### Contexto de negocio

**El cliente:** un e-commerce de tecnologia quiere anticipar qué clientes son propensos a comprar un iPhone.

**El problema:** la frontera entre "compra" y "no compra" no es necesariamente una línea recta — ingresos y fidelidad se combinan de forma no lineal para explicar la decisión de compra.

**La pregunta:** ¿qué kernel de SVM separa mejor a los dos grupos de clientes?

In [ ]:
# Librerías base
import pandas as pd
import numpy as np

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

# Componentes clave de Scikit-Learn para Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

plt.style.use("seaborn-v0_8-whitegrid")
np.random.seed(42)

# Estilo visual — sistema de color validado (consejo UX/UI Data)
BACKGROUND   = '#fbfbfb'
PURPLE       = '#7a7bff'   # único color de énfasis (1 por gráfico)
PURPLE_LIGHT = '#9b9cff'   # EDA de una sola serie (histogramas) -- PURPLE fuerte queda solo para el enfasis
POSITIVE     = '#6b8158'   # exclusivo signo positivo
NEGATIVE     = '#c34031'   # exclusivo signo negativo
NEUTRAL_BAR  = '#d9d9d9'   # barras/áreas de contexto (siempre con etiqueta de valor)
NEUTRAL_LINE = '#8f8c9e'   # líneas de contexto (más contraste que NEUTRAL_BAR)
CONTEXT_LINES = [NEUTRAL_LINE, '#a89a8a', '#7d94a8']   # gama fija para 2+ lineas de contexto en un mismo grafico
INK          = '#111111'
MUTED        = '#707070'

# Alias de compatibilidad con el resto del notebook
GREEN, RED, GRAY = POSITIVE, NEGATIVE, NEUTRAL_BAR

DIVERGING_CMAP = LinearSegmentedColormap.from_list(
    "borja_diverging", ["#c34031", "#e0a89f", "#f0ede8", "#b7c2a9", "#6b8158"]
)
SEQUENTIAL_GREEN = LinearSegmentedColormap.from_list(
    "borja_sequential", [BACKGROUND, POSITIVE]
)

def color_annotations(ax, values, threshold, dark="#ffffff", light=INK):
    """Recolorea el texto de un heatmap celda a celda según su magnitud."""
    for text, value in zip(ax.texts, np.asarray(values).flatten()):
        text.set_color(dark if abs(value) >= threshold else light)

plt.rcParams.update({
    'figure.figsize': (10, 5),
    'figure.dpi': 100,
    'figure.facecolor': BACKGROUND,
    'axes.facecolor': BACKGROUND,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.edgecolor': MUTED,
    'axes.labelcolor': INK,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.titlecolor': INK,
    'xtick.color': MUTED,
    'ytick.color': MUTED,
    'font.family': 'sans-serif',
    'font.size': 10,
    'grid.color': '#f0f0f0',
    'grid.linewidth': 0.5,
})

## 2. Carga de datos

In [2]:
data = pd.read_excel("dataset_clientes_iphone.xlsx")
print(f"Registros: {data.shape[0]} | Columnas: {data.shape[1]}")
data.head()

Registros: 233 | Columnas: 3


,Ingresos_Mensuales,Score_Fidelidad,Compra_iPhone
0,2373.49,95.12,1
1,4353.47,59.27,1
2,3591.95,16.44,0
3,3135.70,16.44,0
4,1630.98,86.75,0


## 3. EDA: Análisis Exploratorio de datos

In [3]:
data.describe().round(2)

,Ingresos_Mensuales,Score_Fidelidad,Compra_iPhone
count,233.00,233.00,233.00
mean,2767.00,52.88,0.36
std,931.66,26.55,0.48
min,1108.91,2.18,0.00
25%,2022.35,30.65,0.00
50%,2835.53,54.12,0.00
75%,3512.35,72.11,1.00
max,4412.95,99.43,1.00


In [4]:
# Comprobar nulos y tipos

print('TIPOS DE DATOS Y VALORES NULOS')
print('─' * 70)
info = pd.DataFrame({
    'Tipo': data.dtypes,
    'Nulos': data.isnull().sum(),
    '% Nulos': (data.isnull().sum() / len(data) * 100).round(2),
    'Únicos': data.nunique()
})
info

TIPOS DE DATOS Y VALORES NULOS
──────────────────────────────────────────────────────────────────────


,Tipo,Nulos,% Nulos,Únicos
Ingresos_Mensuales,float64,0,0.0,225
Score_Fidelidad,float64,0,0.0,147
Compra_iPhone,int64,0,0.0,2


#### Exploración de la variable Target (Satisfecho)

In [5]:
sat_rate = data["Compra_iPhone"].value_counts(normalize=True)
print("Distribución del target:")
print(f"  No compra (0): {sat_rate[0]:.1%}")
print(f"  Compra (1):    {sat_rate[1]:.1%}")

Distribución del target:
  No compra (0): 63.5%
  Compra (1):    36.5%


In [ ]:
# Distribuciones de cada variable

fig, axes = plt.subplots(2, 1, figsize=(13, 8))
fig.suptitle('Distribuciones de las variables', fontsize=14, fontweight='bold', color=INK, y=1.00)

variables = ["Ingresos_Mensuales", "Score_Fidelidad"]
for ax, var in zip(axes.flat, variables):
    ax.hist(data[var], bins=20, color=PURPLE_LIGHT, edgecolor='white', alpha=0.85)
    ax.axvline(data[var].mean(), color=INK, linestyle='--', linewidth=1.5, label=f'Media: {data[var].mean():.1f}')
    ax.axvline(data[var].median(), color=NEUTRAL_LINE, linestyle='--', linewidth=1.5, label=f'Mediana: {data[var].median():.1f}')
    ax.set_title(var, color=INK)
    ax.legend(fontsize=9, frameon=False)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Preparacion de datos

**Estandarizacion obligatoria para SVM.** Los kernels SVM son sensibles a la escala de las features.

In [7]:
X = data[[
    "Score_Fidelidad",
    "Ingresos_Mensuales"
]]

y = data["Compra_iPhone"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    stratify=y,
    random_state=42
)

scaler_X = StandardScaler()

X_train_s = scaler_X.fit_transform(X_train)
X_test_s = scaler_X.transform(X_test)

print(f"Train: {X_train_s.shape[0]} | Test: {X_test_s.shape[0]}")

Train: 174 | Test: 59


- Aplicar estratificado ya que es un ejercicio de clasificación.
- Aplicar normalización ya que disponemos variables de diferentes rangos. 

## 5. Modelo base — SVM lineal

In [8]:
modelo_linear = SVC(
    kernel="linear",
    C=1.0,
    random_state=42
)

modelo_linear.fit(
    X_train_s,
    y_train
)

y_pred_linear = modelo_linear.predict(X_test_s)

print(
    classification_report(
        y_test,
        y_pred_linear,
        target_names=["No compra (0)", "Compra (1)"]
    )
)


               precision    recall  f1-score   support

No compra (0)       0.95      0.95      0.95        37
   Compra (1)       0.91      0.91      0.91        22

     accuracy                           0.93        59
    macro avg       0.93      0.93      0.93        59
 weighted avg       0.93      0.93      0.93        59



In [ ]:
# Visualización de la frontera de decisión

fig, ax = plt.subplots(figsize=(10, 6))

sns.scatterplot(
    data=data,
    x="Ingresos_Mensuales",
    y="Score_Fidelidad",
    hue="Compra_iPhone",
    palette={
        0: NEUTRAL_LINE,
        1: PURPLE
    },
    s=80,
    edgecolor=INK,
    alpha=0.8,
    ax=ax
)


# Coeficientes del SVM
w = modelo_linear.coef_[0]
b = modelo_linear.intercept_[0]

# Valores de ingresos para dibujar la línea
x = np.linspace(
    data["Ingresos_Mensuales"].min(),
    data["Ingresos_Mensuales"].max(),
    300
)

# Convertimos ingresos a escala estandarizada
x_scaled = (
    x - scaler_X.mean_[1]
) / scaler_X.scale_[1]

# Frontera en escala estandarizada
# w[0] · Score + w[1] · Ingresos + b = 0

score_scaled = (
    -w[1] * x_scaled - b
) / w[0]

# Volvemos Score a su escala original
score = (
    score_scaled * scaler_X.scale_[0]
    + scaler_X.mean_[0]
)


# Dibujar línea
ax.plot(
    x,
    score,
    color=INK,
    linewidth=3,
    label="Frontera de decisión",
    zorder=5
)

ax.set_title(
    "Frontera de Decisión de SVM: Propensión de Compra de iPhone",
    fontsize=13,
    fontweight="bold"
)

ax.set_xlabel(
    "Ingresos Mensuales del Cliente (€)",
    fontsize=11
)

ax.set_ylabel(
    "Score de Fidelidad Tecnológica (1-100)",
    fontsize=11
)

ax.legend(title="Resultado")

ax.grid(
    True,
    linestyle=":",
    alpha=0.5
)

plt.tight_layout()
plt.show()

### SVM — Kernel Polinómico

In [10]:
# Construcción del modelo SVM con kernel polinomial

modelo_poly = SVC(
    kernel="poly",
    degree=2,
    C=1.0,
    gamma="scale",
    coef0=1,
    random_state=42
)

modelo_poly.fit(
    X_train_s,
    y_train
)

y_pred_poly = modelo_poly.predict(X_test_s)

print(
    classification_report(
        y_test,
        y_pred_poly,
        target_names=["No compra (0)", "Compra (1)"]
    )
)

               precision    recall  f1-score   support

No compra (0)       0.97      1.00      0.99        37
   Compra (1)       1.00      0.95      0.98        22

     accuracy                           0.98        59
    macro avg       0.99      0.98      0.98        59
 weighted avg       0.98      0.98      0.98        59



In [ ]:
# Visualización del modelo

plt.figure(figsize=(9, 6))

sns.scatterplot(
    data=data,
    x="Ingresos_Mensuales",
    y="Score_Fidelidad",
    hue="Compra_iPhone",
    palette={
        0: NEUTRAL_LINE,
        1: PURPLE
    },
    s=80,
    edgecolor=INK
)

# Obtenemos los límites del gráfico
ax = plt.gca()

xlim = ax.get_xlim()
ylim = ax.get_ylim()

# Creamos una malla de puntos
xx = np.linspace(xlim[0], xlim[1], 300)
yy = np.linspace(ylim[0], ylim[1], 300)

XX, YY = np.meshgrid(xx, yy)

# Convertimos la malla en observaciones
xy = np.c_[XX.ravel(), YY.ravel()]

# Escalamos con el MISMO escalador utilizado
xy_scaled = scaler_X.transform(xy[:, [1, 0]])

# Calculamos la función de decisión
Z = modelo_poly.decision_function(xy_scaled)

# Recuperamos la forma de la malla
Z = Z.reshape(XX.shape)

# Dibujamos la frontera de decisión
ax.contour(
    XX,
    YY,
    Z,
    levels=[0],
    colors=INK,
    linewidths=2.5
)

plt.title(
    "Frontera de Decisión de SVM: Kernel Polinómico",
    fontsize=13,
    fontweight="bold"
)

plt.xlabel(
    "Ingresos Mensuales del Cliente (€)",
    fontsize=11
)

plt.ylabel(
    "Score de Fidelidad Tecnológica (1-100)",
    fontsize=11
)

plt.grid(
    True,
    linestyle=":",
    alpha=0.5
)

plt.legend(title="Compra iPhone")

plt.tight_layout()
plt.show()

#### SVM — Kernel RBF

In [12]:
modelo_rbf = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale",
    random_state=42
)

modelo_rbf.fit(
    X_train_s,
    y_train
)

y_pred_rbf = modelo_rbf.predict(X_test_s)

print(
    classification_report(
        y_test,
        y_pred_rbf,
        target_names=["No compra (0)", "Compra (1)"]
    )
)

               precision    recall  f1-score   support

No compra (0)       0.95      1.00      0.97        37
   Compra (1)       1.00      0.91      0.95        22

     accuracy                           0.97        59
    macro avg       0.97      0.95      0.96        59
 weighted avg       0.97      0.97      0.97        59



In [ ]:
# Visualización de la frontera de decisión

plt.figure(figsize=(9, 6))

sns.scatterplot(
    data=data,
    x="Ingresos_Mensuales",
    y="Score_Fidelidad",
    hue="Compra_iPhone",
    palette={
        0: NEUTRAL_LINE,
        1: PURPLE
    },
    s=80,
    edgecolor=INK
)

# Límites del gráfico
ax = plt.gca()

xlim = ax.get_xlim()
ylim = ax.get_ylim()

# Malla de puntos
xx = np.linspace(xlim[0], xlim[1], 300)
yy = np.linspace(ylim[0], ylim[1], 300)

XX, YY = np.meshgrid(xx, yy)

# Convertimos la malla en observaciones
xy = np.c_[XX.ravel(), YY.ravel()]

# El modelo espera:
# Score_Fidelidad, Ingresos_Mensuales
xy_scaled = scaler_X.transform(xy[:, [1, 0]])

# Función de decisión
Z = modelo_rbf.decision_function(xy_scaled)
Z = Z.reshape(XX.shape)

# Frontera de decisión
ax.contour(
    XX,
    YY,
    Z,
    levels=[0],
    colors=INK,
    linewidths=2.5
)

# Formato
plt.title(
    "Frontera de Decisión de SVM: Kernel RBF",
    fontsize=13,
    fontweight="bold"
)

plt.xlabel(
    "Ingresos Mensuales del Cliente (€)",
    fontsize=11
)

plt.ylabel(
    "Score de Fidelidad Tecnológica (1-100)",
    fontsize=11
)

plt.grid(
    True,
    linestyle=":",
    alpha=0.5
)

plt.legend(title="Compra iPhone")

plt.tight_layout()
plt.show()

## Conclusión

El análisis muestra que la relación entre **ingresos, fidelidad y compra de iPhone no se separa completamente mediante una frontera lineal**.

El **SVM lineal alcanza un 93% de accuracy**, un resultado ya sólido, pero los kernels no lineales lo mejoran: el **kernel polinómico llega al 98%** y el **kernel RBF al 97%**, capturando mejor la estructura no lineal de los clientes.

### Insight de negocio

**El perfil de compra no depende únicamente de los ingresos. La combinación de capacidad económica y fidelidad tecnológica permite identificar mejor a los clientes con mayor propensión a comprar un iPhone.**

Por tanto, para este dataset, el **kernel polinómico (grado 2)** es la opción más adecuada, aunque el modelo debe considerarse ilustrativo porque se trabaja con un dataset sintético y únicamente dos variables predictoras.